# neuralop dataset summary — *verified from the actual loaded tensors*


In [ ]:
import warnings

import numpy as np
import torch

warnings.filterwarnings("ignore")

def report(name, physics, eqn, x, y, n_samples):
    """Print the requested facts straight from the tensors x (B,C,*spatial), y."""
    x, y = torch.as_tensor(x), torch.as_tensor(y)
    B, Cin, *sp = x.shape
    Cout, d, per = y.shape[1], len(sp), int(np.prod(sp))
    print(f"{name}   |   {physics}")
    print(f"  RAW (ground truth) : x {tuple(x.shape)}   y {tuple(y.shape)}")
    print(f"  4) grid dimension  : {d}D,  resolution {tuple(sp)}   channels in={Cin} out={Cout}")
    print(f"  5) PDE solved      : {eqn}")
    print(f"  3) per-batch B     : {B}")
    print(f"  1) total points    : {n_samples} samples x {per} pts/sample = {n_samples*per:,}")
    print("  2) independent 1D FFTs per spatial axis  (K = B x C x other-axes):")
    for i, Ni in enumerate(sp):
        oth = int(np.prod([sp[j] for j in range(d) if j != i]))
        print(f"        axis{i}:  N'={Ni:<5} K = {B}x{Cin}x{oth} = {B*Cin*oth:,}")
    print("     (in-model x hidden_channels; FNO keeps n_modes<=N'/2 of the output)\n")

def mesh_report(name, physics, sample):
    """Unstructured-mesh datasets: dump every tensor field's shape (no separable FFT applies)."""
    print(f"{name}   |   {physics}   (UNSTRUCTURED MESH / point cloud -- no separable FFT)")
    for k, v in sample.items():
        try: print(f"  {k:24} {tuple(torch.as_tensor(v).shape)}")
        except Exception: pass
    print()
print("helpers ready")

helpers ready


## 1. Darcy flow — 2D elliptic PDE  (**FULL**, res 128)

The full Darcy-128 release (`darcy_train_128.pt`). `mmap=True` reads shapes without loading it all; batch = 16 samples + the FNO channel dim.


In [ ]:
DARCY_ROOT = "/data4/home/anirudhgupta/data/darcy"
d = torch.load(f"{DARCY_ROOT}/darcy_train_128.pt", weights_only=False, mmap=True)
x_all, y_all = d["x"], d["y"]                        # (n_train, 128, 128)
print(f"FULL Darcy-128 train file: {x_all.shape[0]:,} samples at {tuple(x_all.shape[1:])}")
xb, yb = x_all[:16].unsqueeze(1), y_all[:16].unsqueeze(1)   # (16,1,128,128): FNO channel_dim=1
report("DARCY-128 (FULL)", "permeability a(x) -> pressure u(x)", "2D second-order elliptic (Darcy's law)",
       xb, yb, x_all.shape[0])

FULL Darcy-128 train file: 5,000 samples at (128, 128)
DARCY-128 (FULL)   |   permeability a(x) -> pressure u(x)
  RAW (ground truth) : x (16, 1, 128, 128)   y (16, 1, 128, 128)
  4) grid dimension  : 2D,  resolution (128, 128)   channels in=1 out=1
  5) PDE solved      : 2D second-order elliptic (Darcy's law)
  3) per-batch B     : 16
  1) total points    : 5000 samples x 16384 pts/sample = 81,920,000
  2) independent 1D FFTs per spatial axis  (K = B x C x other-axes):
        axis0:  N'=128   K = 16x1x128 = 2,048
        axis1:  N'=128   K = 16x1x128 = 2,048
     (in-model x hidden_channels; FNO keeps n_modes<=N'/2 of the output)



## 2. Burgers — 1D space + time  (bundled: res 16)
Input `x` is the **1D initial condition** (16); output `y` is the **space-time solution** (17 time x 16 space) — the field the FNO transforms is 2D.

In [ ]:
from neuralop.data.datasets import load_mini_burgers_1dtime
from neuralop.data.datasets.darcy import example_data_root

tr, te, _ = load_mini_burgers_1dtime(example_data_root, n_train=50, n_test=25,
                                     batch_size=16, test_batch_size=16)
b = next(iter(tr))
print("input x (1D initial condition):", tuple(b["x"].shape),
      "  ->  output y (space-time soln):", tuple(b["y"].shape), "\n")
report("BURGERS (space-time field y)", "viscous Burgers u0(x) -> u(t,x)",
       "1D space + time (FNO sees a 2D t x x grid)", b["y"], b["y"], len(tr.dataset))

Loading test db for resolution 16 with 25 samples 
input x (1D initial condition): (16, 1, 16)   ->  output y (space-time soln): (16, 1, 17, 16) 

BURGERS (space-time field y)   |   viscous Burgers u0(x) -> u(t,x)
  RAW (ground truth) : x (16, 1, 17, 16)   y (16, 1, 17, 16)
  4) grid dimension  : 2D,  resolution (17, 16)   channels in=1 out=1
  5) PDE solved      : 1D space + time (FNO sees a 2D t x x grid)
  3) per-batch B     : 16
  1) total points    : 50 samples x 272 pts/sample = 13,600
  2) independent 1D FFTs per spatial axis  (K = B x C x other-axes):
        axis0:  N'=17    K = 16x1x16 = 256
        axis1:  N'=16    K = 16x1x17 = 272
     (in-model x hidden_channels; FNO keeps n_modes<=N'/2 of the output)



## 3. Navier-Stokes — 2D vorticity  (**FULL** data, res 128)

The full NS-128 release (`nsforcing_train_128.pt`, 1.3 GB, 10,000 samples). Loaded via `mmap=True` so we read shapes without pulling 1.3 GB into RAM. Batch is taken as 8 samples + the FNO channel dim.

In [ ]:
import torch

NS_ROOT = "/data4/home/anirudhgupta/data/navier_stokes"
d = torch.load(f"{NS_ROOT}/nsforcing_train_128.pt", weights_only=False, mmap=True)  # no full RAM load
x_all, y_all = d["x"], d["y"]                       # (10000, 128, 128) each
print(f"FULL NS-128 train file: {x_all.shape[0]:,} samples at {tuple(x_all.shape[1:])}  (config n_train=10000)")
xb = x_all[:8].unsqueeze(1)                         # (8,1,128,128): FNO adds channel_dim=1
yb = y_all[:8].unsqueeze(1)
report("NAVIER-STOKES-128 (FULL)", "2D incompressible NS vorticity w(t) -> w(t+1)",
       "2D Navier-Stokes (time-stepping)", xb, yb, x_all.shape[0])

FULL NS-128 train file: 10,000 samples at (128, 128)  (config n_train=10000)
NAVIER-STOKES-128 (FULL)   |   2D incompressible NS vorticity w(t) -> w(t+1)
  RAW (ground truth) : x (8, 1, 128, 128)   y (8, 1, 128, 128)
  4) grid dimension  : 2D,  resolution (128, 128)   channels in=1 out=1
  5) PDE solved      : 2D Navier-Stokes (time-stepping)
  3) per-batch B     : 8
  1) total points    : 10000 samples x 16384 pts/sample = 163,840,000
  2) independent 1D FFTs per spatial axis  (K = B x C x other-axes):
        axis0:  N'=128   K = 8x1x128 = 1,024
        axis1:  N'=128   K = 8x1x128 = 1,024
     (in-model x hidden_channels; FNO keeps n_modes<=N'/2 of the output)



## 4. ShapeNet-Car — 3D aerodynamics  (bundled mini_car; unstructured mesh)

In [ ]:
from neuralop.data.datasets import load_mini_car

cars = load_mini_car()
print("mini_car: list of", len(cars), "car meshes\n")
mesh_report("SHAPENET-CAR", "3D external car aerodynamics (pressure on surface)", cars[0])
print("-> N = number of mesh vertices (variable per car); 3D surface; NOT an FFT target.")

mini_car: list of 3 car meshes

SHAPENET-CAR   |   3D external car aerodynamics (pressure on surface)   (UNSTRUCTURED MESH / point cloud -- no separable FFT)
  vertices                 (3586, 3)
  vertex_normals           (3586, 3)
  triangle_normals         (7168, 3)
  centroids                (7168, 3)
  triangle_areas           (7168,)
  distance                 (16, 16, 16, 1)
  closest_points           (16, 16, 16, 3)
  normalized_triangle_areas (7168,)
  press                    (1, 3586)
  query_points             (16, 16, 16, 3)

-> N = number of mesh vertices (variable per car); 3D surface; NOT an FFT target.


## 5. OT nonlinear-Poisson  (bundled; mesh / point cloud)

In [ ]:
import torch

d = torch.load("neuralop/data/datasets/data/ot_expand3.0_reg1e-06_train2_test1.pt", weights_only=False)
print("OT nonlinear-Poisson: list of", len(d), "samples")
s = d[0]
if isinstance(s, dict):
    mesh_report("OT-POISSON", "nonlinear Poisson on an optimal-transport mesh", s)
else:
    print("  sample type:", type(s).__name__, "->", getattr(s, "shape", "n/a"))

OT nonlinear-Poisson: list of 3 samples
OT-POISSON   |   nonlinear Poisson on an optimal-transport mesh   (UNSTRUCTURED MESH / point cloud -- no separable FFT)
  target                   (3586, 3)
  source                   (10609, 3)
  ind_enc                  (10609,)
  ind_dec                  (3586,)
  nor_t                    (3586, 3)
  nor_s                    (103, 103, 3)
  trans                    (10609, 3)
  press                    (3682,)



## 6. Spherical Shallow-Water  (needs torch_harmonics; guarded)

In [7]:
try:
    from neuralop.data.datasets import load_spherical_swe
    tr, te = load_spherical_swe(n_train=4, n_tests=[2], batch_size=2, test_batch_sizes=[2],
                                train_resolution=(32, 64), test_resolutions=[(32, 64)])
    b = next(iter(tr))
    report("SPHERICAL-SWE", "shallow water on the sphere", "2D on sphere (spherical harmonics)",
           b["x"], b["y"], len(tr.dataset))
    print("NOTE: SFNO uses a spherical-harmonic transform (SHT), not a plain FFT.")
except Exception as e:
    print("SPHERICAL-SWE skipped (needs torch_harmonics):", repr(e)[:90])
    print("  From config: 2D sphere 256x512, in=3 out=3, B=32, n_modes (16,32); uses SHT not FFT")

SPHERICAL-SWE skipped (needs torch_harmonics): ImportError("cannot import name 'load_spherical_swe' from 'neuralop.data.datasets' (/data4
  From config: 2D sphere 256x512, in=3 out=3, B=32, n_modes (16,32); uses SHT not FFT


## Recap — verified vs FFT-relevance

| dataset | dim | resolution `N'` | in/out ch | grid? | radix-128 fit |
|---|---|---|---|---|---|
| Darcy | 2D | 16 / 32 / (…128/421) | 1 / 1 | ✅ regular | 128 ideal; 16/32 = small leaf |
| Burgers | 2D (t×x) | 17 × 16 (full 101×128) | 1 / 1 | ✅ regular | small leaf; 101/17 not radix-friendly |
| Navier-Stokes | 2D | **128** (or 1024) | 1 / 1 | ✅ regular | **128 = one radix-128 leaf (ideal)** |
| ShapeNet-Car | 3D | variable mesh | — / 4 | ❌ mesh | n/a |
| OT-Poisson | — | mesh / points | — | ❌ mesh | n/a |
| Spherical-SWE | 2D sphere | 256×512 | 3 / 3 | sphere (SHT) | n/a — not FFT |

**Only Darcy and Navier-Stokes are regular grids where the radix-B FFT kernel applies**, and
**NS-128 is the sweet spot** (`N'=128`, one radix-128 leaf, `K` in the tens-of-thousands). Also
note: Darcy & NS load the **same source files** the Transolver repo uses (`piececonst_r421…`,
`NavierStokes_V1e-5…`) — just at different resolutions (Transolver runs Darcy at 85, NS at 64).